# Day 2 — LLMs Deep Dive Exercises
## Industrial AI & LLM Training Program

**Session:** Day 2 — Large Language Models Deep Dive  
**Duration:** 30-minute guided lab (21:50–22:20 WIB)  
**Dataset:** 40 synthetic industrial maintenance/ERP/safety tickets (same as Day 1)  

---

### What we will build tonight

| Section | Task | Output |
|---|---|---|
| 0 — Setup | Multi-provider LLM client | Unified `chat()` function |
| 1 — Tokens | Tokenize & estimate cost | Token counts + cost table |
| 2 — First Call | Chat completions API | Live LLM response |
| 3 — Prompt Engineering | Zero-shot → few-shot → CoT → system prompt | Structured JSON output |
| 4 — Function Calling | Tool schema + parsing | Categorized ticket object |
| 5 — Model Comparison | Benchmark 5 tickets | Latency + token table |
| 6 — Full Pipeline | All 40 tickets → LLM | Crosstab vs. ground truth |
| 7 — Next Steps | Reflect and extend | Day 2→5 connection table |

**Run all cells top-to-bottom.** Anthropic → OpenAI → Ollama → Mock (offline).  
Set `ANTHROPIC_API_KEY` in your environment for best results.

In [ ]:
# CELL 0-A: Install required libraries (run this if you get ImportError below)
# Uncomment and run if needed:

# !pip install anthropic openai tiktoken requests pandas numpy matplotlib python-dotenv

In [9]:
# CELL 0-B: Imports and provider detection
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import tiktoken
import requests
warnings.filterwarnings('ignore')

# ── Load .env file (if present) ───────────────────────────────────────────────
# Participants who store API keys in a .env file need python-dotenv to load them.
# If the library isn't installed the notebook still works — keys set via
# `export` in the shell or Jupyter's environment are picked up automatically.
try:
    from dotenv import load_dotenv
    if load_dotenv():          # returns True when a .env file is found and loaded
        print('✓ .env file loaded')
    else:
        print('  (No .env file found — using shell environment variables)')
except ImportError:
    print('  (python-dotenv not installed — run: pip install python-dotenv)')
    print('  Alternatively, set keys with: export ANTHROPIC_API_KEY=your_key')

# ── Provider Detection ───────────────────────────────────────────────────────────
PROVIDER = None
client = None
DEFAULT_MODEL = None

# 1. Try Anthropic
if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        import anthropic
        client = anthropic.Anthropic()
        PROVIDER = 'anthropic'
        DEFAULT_MODEL = 'claude-haiku-4-5-20251001'
        print(f'\u2713 Provider: Anthropic  |  Model: {DEFAULT_MODEL}')
    except Exception as e:
        print(f'Anthropic import failed: {e}')

# 2. Try OpenAI
if PROVIDER is None and os.environ.get('OPENAI_API_KEY'):
    try:
        import openai
        client = openai.OpenAI()
        PROVIDER = 'openai'
        DEFAULT_MODEL = 'gpt-3.5-turbo'
        print(f'\u2713 Provider: OpenAI  |  Model: {DEFAULT_MODEL}')
    except Exception as e:
        print(f'OpenAI import failed: {e}')

# 3. Try Ollama (local server)
if PROVIDER is None:
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            PROVIDER = 'ollama'
            DEFAULT_MODEL = 'llama3.2'
            print(f'\u2713 Provider: Ollama (local)  |  Model: {DEFAULT_MODEL}')
    except Exception:
        pass

# 4. Mock fallback — always succeeds, no setup required
if PROVIDER is None:
    PROVIDER = 'mock'
    DEFAULT_MODEL = 'mock-keyword-v1'
    print('\u26a0  Provider: Mock (offline) | Model: mock-keyword-v1')
    print('   No real LLM detected. Running in MOCK MODE — keyword-based simulation.')
    print('   Accuracy ~85-90% on these keyword-rich tickets (vs ~95% for real LLMs).')
    print('   On real-world tickets without clear keywords, mock accuracy drops sharply.')
    print('   To use a real model: set ANTHROPIC_API_KEY or start Ollama.')

if PROVIDER:
    print(f'\nAll imports successful! Using {PROVIDER} provider.')

✓ .env file loaded
✓ Provider: Anthropic  |  Model: claude-haiku-4-5-20251001

All imports successful! Using anthropic provider.


In [10]:
# CELL 0-C: Unified chat() function — works with Anthropic, OpenAI, Ollama, or Mock

# ── Mock LLM ──────────────────────────────────────────────────────────────────
def _mock_llm(messages, system=''):
    """
    Keyword-based mock LLM. Returns plausible JSON for triage prompts.
    Used when no real provider is available (offline / no API key).

    Accuracy: ~65-82% on the 40 known tickets.
    The intentional gap vs real LLMs (~95%) is the pedagogical point.
    """
    # Combine user message text for scanning (do NOT include system — it contains
    # category names like "SAP/ERP" that would bias keyword scores)
    ticket_text = ' '.join(
        m.get('content', '') for m in messages
    ).lower()

    # Keep full combined text for canned-response heuristics only
    all_text = ticket_text + ' ' + system.lower()

    # ── Canned responses for non-triage cells ──────────────────────────────
    is_triage = any(cat.lower() in system.lower()
                    for cat in ['mechanical', 'sap/erp', 'network/it', 'safety'])

    if 'loto' in all_text and 'stand for' in all_text:
        return (
            "LOTO stands for Lockout/Tagout. It is a critical safety procedure that "
            "ensures hazardous energy sources (electrical, hydraulic, pneumatic, chemical) "
            "are isolated and de-energized before maintenance begins. Workers attach a "
            "personal lock to the energy isolation point so no one can re-energize the "
            "equipment while someone is working on it. LOTO prevents an estimated 120 "
            "fatalities and 50,000 injuries in the US alone each year."
        )

    if 'bearing failure' in all_text or ('bearing' in all_text and 'failure mode' in all_text):
        return (
            "Three common bearing failure modes in industrial pumps:\n"
            "1. Fatigue spalling — surface cracks from cyclic stress, appears as pitting on races\n"
            "2. Lubrication failure — inadequate or contaminated grease/oil causes metal-to-metal contact\n"
            "3. Misalignment — shaft or housing misalignment generates abnormal radial/axial loads"
        )

    if not is_triage:
        return (
            "[MOCK MODE — offline] This is a simulated response. "
            "Connect to a real LLM provider for actual model output."
        )

    # ── Keyword scoring for triage ─────────────────────────────────────────
    safety_kw    = ['loto', 'apd', 'spill', 'near miss', 'kebakaran', 'kontraktor',
                    'tangga', 'rambu', 'nitrogen', 'hot work', 'welding shield',
                    'permit', 'izin kerja', 'guard', 'false alarm']
    sap_kw       = ['me21n', 'migo', 'miro', 'purchase order', 'iw51', 'co01',
                    's alr', 'workflow', 'invoice', 'goods receipt', 'vendor',
                    'mm01', 'mb51', 'transaction code', 'erp', 'sap']
    network_kw   = ['scada', 'server', 'network', 'wifi', 'vpn', 'firewall',
                    'ntp', 'cctv', 'backup', 'printer', 'firmware', 'plc',
                    'historian', 'pi ', 'switch', 'access point', 'email server',
                    'database backup', 'ntp', 'firewall']
    mechanical_kw = ['pompa', 'pump', 'kompresor', 'bearing', 'vibrasi', 'motor',
                     'valve', 'belt', 'gearbox', 'coupling', 'fan', 'seal',
                     'actuator', 'heat exchanger', 'vacuum', 'shaft', 'blade',
                     'fouling', 'cooling tower', 'overheating', 'alignment']

    scores = {
        'Safety':     sum(1 for kw in safety_kw     if kw in ticket_text),
        'SAP/ERP':    sum(1 for kw in sap_kw        if kw in ticket_text),
        'Network/IT': sum(1 for kw in network_kw    if kw in ticket_text),
        'Mechanical': sum(1 for kw in mechanical_kw if kw in ticket_text),
    }

    # Pick category with highest score; default to Mechanical on tie
    category = max(scores, key=lambda k: (scores[k], k == 'Mechanical'))

    # ── Priority heuristics ────────────────────────────────────────────────
    critical_kw = ['loto', 'shutdown', 'bahaya', 'near miss', 'kebakaran',
                   'blade retak', 'gas nitrogen', 'tangki']
    high_kw     = ['kebocoran', 'tidak bisa start', 'data gap', 'tidak sinkron',
                   'overheating', 'retak', 'stuck']

    if any(kw in ticket_text for kw in critical_kw):
        priority = 'Critical'
    elif any(kw in ticket_text for kw in high_kw):
        priority = 'High'
    else:
        priority = 'Medium'

    # ── Build reason ───────────────────────────────────────────────────────
    reasons = {
        'Safety':     'Ticket contains safety-related keywords (LOTO/APD/permit/spill).',
        'SAP/ERP':    'Ticket references SAP transactions or ERP workflow issues.',
        'Network/IT': 'Ticket involves network infrastructure, servers, or connectivity.',
        'Mechanical': 'Ticket describes equipment failure or mechanical system issue.',
    }

    result = {
        'category': category,
        'priority': priority,
        'reason': reasons[category],
    }
    return json.dumps(result)


# ── Unified chat() ───────────────────────────────────────────────────────────────
def chat(messages, system=None, model=None, temperature=0.0):
    """
    Unified LLM call interface across providers.

    Args:
        messages (list): List of dicts with 'role' and 'content' keys.
                         Roles: 'user' | 'assistant'
        system (str): System prompt (optional)
        model (str): Override model name (defaults to DEFAULT_MODEL)
        temperature (float): Sampling temperature. 0.0 = deterministic.

    Returns:
        str: The model's text response
    """
    model = model or DEFAULT_MODEL

    if PROVIDER == 'anthropic':
        kwargs = dict(
            model=model,
            max_tokens=1024,
            messages=messages,
            temperature=temperature,
        )
        if system:
            kwargs['system'] = system
        response = client.messages.create(**kwargs)
        return response.content[0].text

    elif PROVIDER == 'openai':
        all_messages = []
        if system:
            all_messages.append({'role': 'system', 'content': system})
        all_messages.extend(messages)
        response = client.chat.completions.create(
            model=model,
            messages=all_messages,
            temperature=temperature,
            max_tokens=1024,
        )
        return response.choices[0].message.content

    elif PROVIDER == 'ollama':
        all_messages = []
        if system:
            all_messages.append({'role': 'system', 'content': system})
        all_messages.extend(messages)
        payload = {
            'model': model,
            'messages': all_messages,
            'stream': False,
            'options': {'temperature': temperature},
        }
        r = requests.post('http://localhost:11434/api/chat', json=payload, timeout=60)
        r.raise_for_status()
        return r.json()['message']['content']

    elif PROVIDER == 'mock':
        return _mock_llm(messages, system or '')

    else:
        raise RuntimeError('No provider configured. Run the imports cell first.')


# ── Smoke test ───────────────────────────────────────────────────────────────────
if PROVIDER == 'mock':
    print('\u26a0  MOCK MODE — offline, educational only')
    print('   Keyword-based simulation active. No API calls will be made.')
    print()
    # Quick sanity check: classify a known safety ticket
    test_resp = chat(
        messages=[{'role': 'user', 'content': 'Ticket: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan'}],
        system='You are an industrial support triage assistant. Categorize tickets into: Mechanical, SAP/ERP, Network/IT, or Safety. Respond ONLY with valid JSON: {"category": "...", "priority": "...", "reason": "..."}',
    )
    print(f'Mock smoke test (K05 — should be Safety/Critical):')
    print(f'  Raw: {test_resp}')
    try:
        parsed = json.loads(test_resp)
        print(f'  Parsed: category={parsed["category"]}, priority={parsed["priority"]}')
    except Exception:
        print('  (not JSON — canned response mode)')
else:
    response = chat(
        messages=[{'role': 'user', 'content': 'Hello! Reply in exactly one sentence.'}],
        system='You are a helpful assistant.',
    )
    print(f'Provider: {PROVIDER}')
    print(f'Model:    {DEFAULT_MODEL}')
    print(f'Response: {response}')


Provider: anthropic
Model:    claude-haiku-4-5-20251001
Response: Hello! I'm happy to help you with whatever you need today.


---
## Section 1 — Tokens: What LLMs Actually See

LLMs do not read words. They read **tokens** — chunks of text produced by a subword tokenizer.
Tokens are the unit of input and output, and they directly determine:
- **Cost** — you pay per input token and per output token
- **Speed** — more tokens = longer processing time
- **Context limit** — each model has a maximum context window measured in tokens

### How Byte-Pair Encoding (BPE) works

BPE tokenization starts with individual characters and iteratively merges the most frequent adjacent pairs into a single token:
1. Start: `['p', 'o', 'm', 'p', 'a']`
2. Merge most common pair: `['po', 'm', 'p', 'a']`
3. Continue merging: `['pom', 'pa']` → `['pompa']`

Common English words become single tokens. Rare words, technical terms, and non-English text get split into multiple subword tokens.

### Why Indonesian costs more than English

Most LLMs are trained on predominantly English text. BPE learns common English subwords during training. Indonesian words — even common ones — are rarer in the training corpus, so they get split into more pieces. The practical impact: the same meaning expressed in Indonesian typically uses **~30% more tokens** than in English — and therefore costs ~30% more to process at scale.

### Design decisions

**Why `tiktoken` and not the Anthropic tokenizer?**  
Anthropic does not expose a standalone Python tokenizer library. `tiktoken` implements the same BPE algorithm used by OpenAI's GPT models. The token counts are slightly different from Anthropic's Claude (Claude uses a different vocabulary), but the orders of magnitude and the Indonesian-vs-English ratio observation hold across providers. For exact Claude token counts, use `response.usage.input_tokens` from the API response object.

**Why `cl100k_base` encoding?**  
This is the BPE vocabulary used by `text-embedding-ada-002`, `gpt-3.5-turbo`, and `gpt-4`. With 100,000 merge rules, it is one of the most capable general-purpose subword vocabularies available.

In [11]:
# CELL 1-A: Tokenization demo — see what the model actually sees

enc = tiktoken.get_encoding('cl100k_base')

demo_texts = [
    ('Indonesian', 'Pompa sentrifugal P-101 vibrasi berlebihan, perlu ganti bearing segera'),
    ('English',    'Centrifugal pump P-101 excessive vibration, need to replace bearing immediately'),
    ('Mixed',      'Pump P-101 vibrasi berlebihan \u2014 bearing replacement needed segera'),
]

print('\u2500\u2500 Tokenization Demo \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
for lang, text in demo_texts:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f'\n[{lang}] ({len(tokens)} tokens)')
    print(f'  Text:    {text}')
    print(f'  Tokens:  {tokens[:15]}...' if len(tokens) > 15 else f'  Tokens:  {tokens}')
    print(f'  Decoded: {decoded}')

# Indonesian vs English token ratio
id_tokens = len(enc.encode(demo_texts[0][1]))
en_tokens = len(enc.encode(demo_texts[1][1]))
print(f'\n\u2500\u2500 Token Count Comparison \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Indonesian: {id_tokens} tokens')
print(f'  English:    {en_tokens} tokens')
print(f'  Ratio:      {id_tokens/en_tokens:.2f}x  ({(id_tokens/en_tokens - 1)*100:.0f}% more tokens for same meaning)')

── Tokenization Demo ──────────────────────────────────────────────

[Indonesian] (24 tokens)
  Text:    Pompa sentrifugal P-101 vibrasi berlebihan, perlu ganti bearing segera
  Tokens:  [47, 316, 6733, 3288, 93366, 45284, 393, 12, 4645, 17358, 10426, 10418, 273, 41216, 276]...
  Decoded: ['P', 'om', 'pa', ' sent', 'rif', 'ugal', ' P', '-', '101', ' vibr', 'asi', ' ber', 'le', 'bih', 'an', ',', ' per', 'lu', ' g', 'anti', ' bearing', ' se', 'ger', 'a']

[English] (15 tokens)
  Text:    Centrifugal pump P-101 excessive vibration, need to replace bearing immediately
  Tokens:  [23026, 93366, 45284, 14155, 393, 12, 4645, 27639, 48913, 11, 1205, 311, 8454, 18534, 7214]
  Decoded: ['Cent', 'rif', 'ugal', ' pump', ' P', '-', '101', ' excessive', ' vibration', ',', ' need', ' to', ' replace', ' bearing', ' immediately']

[Mixed] (18 tokens)
  Text:    Pump P-101 vibrasi berlebihan — bearing replacement needed segera
  Tokens:  [47, 1538, 393, 12, 4645, 17358, 10426, 10418, 273, 41216, 276, 20

In [12]:
# CELL 1-B: Load 40 industrial tickets and estimate API cost

# ── 40 synthetic industrial tickets (same as Day 1) ────────────────────────────────────
tickets_raw = [
    # MECHANICAL
    {'id': 'M01', 'category': 'Mechanical', 'text': 'Pompa sentrifugal P-101 vibrasi berlebihan, perlu ganti bearing segera sebelum shutdown'},
    {'id': 'M02', 'category': 'Mechanical', 'text': 'Kompresor K-202 shutdown otomatis karena overheating, suhu mencapai 95 derajat Celsius'},
    {'id': 'M03', 'category': 'Mechanical', 'text': 'Kebocoran oli pada gearbox GB-103, seal sudah aus perlu penggantian segera'},
    {'id': 'M04', 'category': 'Mechanical', 'text': 'Motor listrik M-305 tidak bisa start, kemungkinan winding terbakar perlu megger test'},
    {'id': 'M05', 'category': 'Mechanical', 'text': 'Valve control HV-201 macet tidak bisa fully open saat startup, actuator bermasalah'},
    {'id': 'M06', 'category': 'Mechanical', 'text': 'Belt conveyor BC-101 slip dan bunyi aneh, perlu alignment dan cek kondisi belt'},
    {'id': 'M07', 'category': 'Mechanical', 'text': 'Heat exchanger HE-302 fouling parah efisiensi turun 30 persen perlu chemical cleaning'},
    {'id': 'M08', 'category': 'Mechanical', 'text': 'Pompa vacuum VP-105 tidak mencapai target vacuum ada kebocoran di flange connection'},
    {'id': 'M09', 'category': 'Mechanical', 'text': 'Coupling antara motor dan pompa P-204 rusak vibrasi tinggi perlu penggantian coupling'},
    {'id': 'M10', 'category': 'Mechanical', 'text': 'Fan cooling tower CT-101 blade retak bahaya jika dibiarkan beroperasi perlu shutdown'},
    # SAP/ERP
    {'id': 'S01', 'category': 'SAP/ERP', 'text': 'Error ME21N saat buat purchase order vendor master belum disetujui procurement department'},
    {'id': 'S02', 'category': 'SAP/ERP', 'text': 'Goods receipt MIGO tidak bisa diposting dokumen PO sudah closed perlu reopen PO'},
    {'id': 'S03', 'category': 'SAP/ERP', 'text': 'MIRO invoice verification gagal amount mismatch dengan PO perlu koordinasi finance'},
    {'id': 'S04', 'category': 'SAP/ERP', 'text': 'User tidak bisa akses transaction code MM01 setelah role change oleh IT admin'},
    {'id': 'S05', 'category': 'SAP/ERP', 'text': 'Batch job MB51 gagal tengah malam material document tidak ter-generate perlu rerun'},
    {'id': 'S06', 'category': 'SAP/ERP', 'text': 'Plant maintenance notification IW51 tidak bisa disimpan mandatory field kosong'},
    {'id': 'S07', 'category': 'SAP/ERP', 'text': 'SAP production order CO01 error karena bill of material tidak aktif perlu aktivasi'},
    {'id': 'S08', 'category': 'SAP/ERP', 'text': 'Report S ALR 87013019 timeout untuk period Q3 data terlalu besar perlu optimasi query'},
    {'id': 'S09', 'category': 'SAP/ERP', 'text': 'SAP login sangat lambat sejak weekend maintenance response time lebih dari 30 detik'},
    {'id': 'S10', 'category': 'SAP/ERP', 'text': 'Workflow approval purchase order stuck di inbox manager sedang cuti perlu delegate'},
    # NETWORK/IT
    {'id': 'N01', 'category': 'Network/IT', 'text': 'SCADA server tidak bisa connect ke PLC area A network timeout perlu cek switch'},
    {'id': 'N02', 'category': 'Network/IT', 'text': 'Printer di control room offline setelah firmware update perlu rollback driver'},
    {'id': 'N03', 'category': 'Network/IT', 'text': 'WiFi area gudang intermittent operator tidak bisa scan barcode perlu cek access point'},
    {'id': 'N04', 'category': 'Network/IT', 'text': 'VPN connection ke kantor pusat putus setiap 2 jam perlu reset manual oleh IT'},
    {'id': 'N05', 'category': 'Network/IT', 'text': 'Database backup server storage penuh backup gagal 3 hari berturut-turut perlu cleanup'},
    {'id': 'N06', 'category': 'Network/IT', 'text': 'Email server bounce semua attachment lebih dari 5MB sejak kemarin perlu cek konfigurasi'},
    {'id': 'N07', 'category': 'Network/IT', 'text': 'CCTV di area produksi 4 kamera offline sekaligus kemungkinan switch port rusak'},
    {'id': 'N08', 'category': 'Network/IT', 'text': 'Server historian OSIsoft PI tidak sinkron dengan DCS data gap 6 jam perlu resync'},
    {'id': 'N09', 'category': 'Network/IT', 'text': 'Firewall block akses ke supplier portal setelah security policy update perlu whitelist'},
    {'id': 'N10', 'category': 'Network/IT', 'text': 'NTP server drift timestamp PLC berbeda 15 menit dari server SAP perlu sinkronisasi'},
    # SAFETY
    {'id': 'K01', 'category': 'Safety', 'text': 'Near miss operator hampir tertimpa material jatuh dari rak gudang perlu pasang guard'},
    {'id': 'K02', 'category': 'Safety', 'text': 'APD tidak tersedia di area welding pekerja memakai safety glasses biasa bukan welding shield'},
    {'id': 'K03', 'category': 'Safety', 'text': 'Spill oli di area pompa belum dibersihkan setelah 2 jam risiko terpeleset sangat tinggi'},
    {'id': 'K04', 'category': 'Safety', 'text': 'Izin kerja panas hot work permit tidak ditandatangani sebelum pengelasan dimulai'},
    {'id': 'K05', 'category': 'Safety', 'text': 'Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan'},
    {'id': 'K06', 'category': 'Safety', 'text': 'Alarm kebakaran berbunyi di area laboratorium ternyata false alarm dari sensor debu'},
    {'id': 'K07', 'category': 'Safety', 'text': 'Tangga portable rusak satu anak tangga masih digunakan pekerja perlu segera diganti'},
    {'id': 'K08', 'category': 'Safety', 'text': 'Papan rambu bahaya listrik hilang di panel MCC-102 perlu pasang rambu baru segera'},
    {'id': 'K09', 'category': 'Safety', 'text': 'Pekerja kontraktor bekerja tanpa safety induction yang valid perlu hentikan pekerjaan'},
    {'id': 'K10', 'category': 'Safety', 'text': 'Tabung gas nitrogen di area proses tidak diikat ke dinding risiko jatuh dan kebocoran'},
]

df = pd.DataFrame(tickets_raw)

# ── Token counting ──────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = 'You are an industrial support triage assistant. Categorize tickets into: Mechanical, SAP/ERP, Network/IT, or Safety.'

def count_prompt_tokens(ticket_text):
    full_prompt = SYSTEM_PROMPT + '\n\nTicket: ' + ticket_text + '\n\nCategory:'
    return len(enc.encode(full_prompt))

df['input_tokens_est'] = df['text'].apply(count_prompt_tokens)
total_tokens = df['input_tokens_est'].sum()

# ── Pricing table (per 1M input tokens) ──────────────────────────────────────────────
PRICING = {
    'Claude Haiku 4.5':   0.25,
    'Claude Sonnet 4.6':  3.00,
    'GPT-3.5 Turbo':      0.50,
    'GPT-4o':            2.50,
    'Mock (offline)':     0.00,
}

print(f'\u2500\u2500 Token Cost Estimation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Total tickets:          {len(df)}')
print(f'Avg tokens per ticket:  {df["input_tokens_est"].mean():.0f}')
print(f'Total input tokens:     {total_tokens:,}')
print()
print(f'{"Model":<22} {"$/1M tok":>10} {"Cost (40 tickets)":>20} {"Cost (10k tickets)":>22}')
print('-' * 80)
for model_name, price_per_m in PRICING.items():
    if price_per_m == 0.00:
        print(f'{model_name:<22} {"Free":>10} {"$0.000000":>20} {"$0.0000":>22}')
        print(f'  \u2514 0 tokens sent \u2014 all computation is local')
    else:
        cost_40   = (total_tokens / 1_000_000) * price_per_m
        cost_10k  = (total_tokens / 40) * 10_000 / 1_000_000 * price_per_m
        print(f'{model_name:<22} ${price_per_m:>9.2f} ${cost_40:>19.6f} ${cost_10k:>21.4f}')

print(f'\n>>> Haiku makes this lab essentially free. Switch to Sonnet for higher quality.')


── Token Cost Estimation ────────────────────────────────────────────
Total tickets:          40
Avg tokens per ticket:  54
Total input tokens:     2,140

Model                    $/1M tok    Cost (40 tickets)     Cost (10k tickets)
--------------------------------------------------------------------------------
Claude Haiku 4.5       $     0.25 $           0.000535 $               0.1338
Claude Sonnet 4.6      $     3.00 $           0.006420 $               1.6050
GPT-3.5 Turbo          $     0.50 $           0.001070 $               0.2675
GPT-4o                 $     2.50 $           0.005350 $               1.3375
Mock (offline)               Free            $0.000000                $0.0000
  └ 0 tokens sent — all computation is local

>>> Haiku makes this lab essentially free. Switch to Sonnet for higher quality.


---
## Section 2 — Your First LLM API Call

The chat completions API is the universal interface to modern LLMs. Understanding its anatomy unlocks every provider.

### The three message roles

| Role | Purpose | Who sets it |
|---|---|---|
| `system` | Persistent instructions; sets model persona, constraints, output format | You (developer) |
| `user` | The human's input turn | User / your application |
| `assistant` | The model's previous response | Model (or you, in few-shot) |

The model processes these in sequence, building a conversational context. The `system` message is processed first and establishes the baseline behavior for the entire conversation.

### Temperature — the creativity dial

- **0.0** — Deterministic. The model picks the highest-probability token at every step. Best for factual tasks, classification, structured output.
- **0.3–0.7** — Balanced. Introduces some variety while remaining coherent. Good for summarization, Q&A.
- **1.0+** — Creative. High variance. Good for brainstorming, creative writing. Bad for structured output.

For industrial triage and classification: always use **temperature=0.0**.

### The response object

```python
# Anthropic response fields
response.content[0].text   # the model's reply (string)
response.model             # actual model used
response.usage.input_tokens  # exact input token count (billed)
response.usage.output_tokens # exact output token count (billed)
response.stop_reason       # 'end_turn' | 'max_tokens' | 'stop_sequence'
```

In [16]:
# CELL 2-A: First LLM call — industrial domain question

response_text = chat(
    messages=[
        {
            'role': 'user',
            'content': 'What does LOTO stand for in a manufacturing context, and why is it critical for worker safety?'
        }
    ],
    system='You are an industrial safety expert. Be concise and practical.',
    temperature=0.0,
)

print('\u2500\u2500 First LLM Call \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Provider: {PROVIDER}  |  Model: {DEFAULT_MODEL}')
print()
print('Question: What does LOTO stand for in a manufacturing context?')
print()
print('Answer:')
print(response_text)

── First LLM Call ─────────────────────────────────────────────────
Provider: anthropic  |  Model: claude-haiku-4-5-20251001

Question: What does LOTO stand for in a manufacturing context?

Answer:
# LOTO: Lockout/Tagout

## What It Stands For
**Lockout/Tagout** – a safety procedure that physically prevents machinery from operating during maintenance or repair.

## How It Works
- **Lockout**: Physical locks disable equipment (electrical switches, valves, etc.)
- **Tagout**: Warning tags alert workers that equipment is disabled
- Only authorized personnel can remove locks/tags

## Why It's Critical

**Prevents Serious Injuries:**
- Stops unexpected machine startup during maintenance
- Protects against electrical shock, crushing, amputation, and burns
- Eliminates the most common cause of maintenance-related deaths

**Legal Requirement:**
- OSHA mandates LOTO procedures (29 CFR 1910.147)
- Violations carry significant penalties

**Real-World Impact:**
- ~120 workers die annually from fai

In [17]:
# CELL 2-B: Inspect the full response object and usage metadata

if PROVIDER == 'anthropic':
    raw = client.messages.create(
        model=DEFAULT_MODEL,
        max_tokens=256,
        messages=[{'role': 'user', 'content': 'Name three common bearing failure modes in industrial pumps. Be brief.'}],
        system='You are an industrial maintenance expert.',
        temperature=0.0,
    )
    print('\u2500\u2500 Full Response Object (Anthropic) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    print(f'  model:              {raw.model}')
    print(f'  stop_reason:        {raw.stop_reason}')
    print(f'  usage.input_tokens: {raw.usage.input_tokens}')
    print(f'  usage.output_tokens:{raw.usage.output_tokens}')
    print(f'  content[0].type:    {raw.content[0].type}')
    print()
    print('  Response text:')
    print(raw.content[0].text)
    print()
    cost = (raw.usage.input_tokens + raw.usage.output_tokens) / 1_000_000 * 0.25
    print(f'  Exact cost for this call (Haiku): ${cost:.8f}')

elif PROVIDER == 'openai':
    raw = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are an industrial maintenance expert.'},
            {'role': 'user', 'content': 'Name three common bearing failure modes in industrial pumps. Be brief.'}
        ],
        temperature=0.0,
        max_tokens=256,
    )
    print('\u2500\u2500 Full Response Object (OpenAI) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    print(f'  model:                  {raw.model}')
    print(f'  finish_reason:          {raw.choices[0].finish_reason}')
    print(f'  usage.prompt_tokens:    {raw.usage.prompt_tokens}')
    print(f'  usage.completion_tokens:{raw.usage.completion_tokens}')
    print()
    print('  Response text:')
    print(raw.choices[0].message.content)

elif PROVIDER == 'mock':
    mock_resp = chat(
        messages=[{'role': 'user', 'content': 'Name three common bearing failure modes in industrial pumps. Be brief.'}],
        system='You are an industrial maintenance expert.',
    )
    print('\u2500\u2500 Full Response Object (Mock) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    print(f'  model:              mock-keyword-v1')
    print(f'  stop_reason:        end_turn (simulated)')
    print(f'  usage.input_tokens: N/A (mock \u2014 no API call made)')
    print(f'  usage.output_tokens: N/A')
    print(f'  note: Connect to a real provider to see actual token counts and billing.')
    print()
    print('  Response text:')
    print(mock_resp)

else:  # ollama
    result = chat(
        messages=[{'role': 'user', 'content': 'Name three common bearing failure modes in industrial pumps. Be brief.'}],
        system='You are an industrial maintenance expert.',
    )
    print('\u2500\u2500 Response (Ollama) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    print('  Note: Ollama does not expose token counts in the standard interface.')
    print('  Response text:')
    print(result)


── Full Response Object (Anthropic) ────────────────────────────────────────────
  model:              claude-haiku-4-5-20251001
  stop_reason:        end_turn
  usage.input_tokens: 28
  usage.output_tokens:120
  content[0].type:    text

  Response text:
# Three Common Bearing Failure Modes in Industrial Pumps

1. **Fatigue Spalling** – Surface material flakes off due to repeated stress cycles, causing pitting on raceways and rolling elements.

2. **Lubrication Failure** – Inadequate or contaminated lubricant leads to metal-to-metal contact, friction, and accelerated wear.

3. **Misalignment** – Shaft or housing misalignment causes uneven load distribution, edge loading, and premature bearing wear.

  Exact cost for this call (Haiku): $0.00003700


---
## Section 3 — Prompt Engineering: The 4-Stage Progression

Prompt engineering is the practice of designing inputs that reliably elicit the output you need.
We use ticket K05 as our running example throughout this section:

> *"Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan"*

**Stage 1 — Zero-shot:** Ask directly with no examples. Simple, but output is unpredictable in format.

**Stage 2 — Few-shot:** Provide 1–4 labelled examples in the prompt. The model learns the output pattern from examples without any fine-tuning. This is in-context learning.

**Stage 3 — Chain-of-thought (CoT):** Ask the model to reason step by step before answering. Particularly effective for ambiguous tickets that could belong to multiple categories.

**Stage 4 — System prompt design:** Assign a specific role, define output format (JSON), and set explicit constraints. This produces machine-parseable, consistent output.

### Why the progression matters

Each stage solves a specific weakness of the previous one:
- Zero-shot → inconsistent format; no guarantee of valid category names
- Few-shot → fixes format; still may lack reasoning for edge cases
- CoT → adds reasoning; output still free-form
- System prompt → constrains format to JSON; enables downstream parsing

### Design decision: temperature=0.0 throughout

Classification is a **factual, deterministic task**. We want the same ticket to always get the same category. Temperature=0.0 makes the model always pick the highest-probability token, producing consistent, reproducible results.

In [18]:
# CELL 3-A: Zero-shot prompting — no examples, just ask

TICKET_K05 = df[df['id'] == 'K05']['text'].values[0]

zero_shot_response = chat(
    messages=[{
        'role': 'user',
        'content': f'Categorize this maintenance ticket into one of these categories: Mechanical, SAP/ERP, Network/IT, Safety.\n\nTicket: {TICKET_K05}'
    }],
    temperature=0.0,
)

print('\u2500\u2500 Stage 1: Zero-Shot \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Ticket [K05]: {TICKET_K05}')
print()
print('Prompt: Categorize this ticket into: Mechanical, SAP/ERP, Network/IT, Safety.')
print()
print('LLM Response:')
print(zero_shot_response)
print()
print('Observation: Output is correct but format is unpredictable.')
print('The model may return a sentence, just the category word, or a paragraph.')

── Stage 1: Zero-Shot ───────────────────────────────────────────────────
Ticket [K05]: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan

Prompt: Categorize this ticket into: Mechanical, SAP/ERP, Network/IT, Safety.

LLM Response:
# Categorization: **Safety**

**Reason:** This ticket is reporting a safety procedure violation. The ticket states that LOTO (Lockout/Tagout) procedure was not performed before technicians entered a cleaning tank. This is a critical safety issue related to hazardous energy control and worker protection, not a mechanical repair, system software issue, or network problem.

Observation: Output is correct but format is unpredictable.
The model may return a sentence, just the category word, or a paragraph.


In [19]:
# CELL 3-B: Few-shot prompting — provide labelled examples in the prompt

few_shot_examples = [
    ('Pompa sentrifugal P-101 vibrasi berlebihan, perlu ganti bearing segera', 'Mechanical'),
    ('Error ME21N saat buat purchase order vendor master belum disetujui', 'SAP/ERP'),
    ('SCADA server tidak bisa connect ke PLC area A network timeout perlu cek switch', 'Network/IT'),
    ('Near miss operator hampir tertimpa material jatuh dari rak gudang', 'Safety'),
]

# Build few-shot prompt
examples_text = '\n'.join(
    f'Ticket: {text}\nCategory: {label}' for text, label in few_shot_examples
)

few_shot_prompt = f"""Categorize each ticket into: Mechanical, SAP/ERP, Network/IT, or Safety.
Reply with ONLY the category name.

{examples_text}

Ticket: {TICKET_K05}
Category:"""

few_shot_response = chat(
    messages=[{'role': 'user', 'content': few_shot_prompt}],
    temperature=0.0,
)

print('\u2500\u2500 Stage 2: Few-Shot \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Ticket [K05]: {TICKET_K05}')
print()
print('Prompt includes 4 labelled examples (one per category).')
print()
print('LLM Response:')
print(few_shot_response)
print()
print('Improvement: Output is now constrained to the category name. Format is reliable.')

── Stage 2: Few-Shot ────────────────────────────────────────────────────
Ticket [K05]: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan

Prompt includes 4 labelled examples (one per category).

LLM Response:
Safety

Improvement: Output is now constrained to the category name. Format is reliable.


In [20]:
# CELL 3-C: Chain-of-thought — ask the model to reason before answering

cot_prompt = f"""Categorize this industrial ticket. Think step by step:
1. What is the main subject of the ticket? (equipment, system, person, process)
2. What keywords indicate the category?
3. What is the final category?

Categories: Mechanical, SAP/ERP, Network/IT, Safety

Ticket: {TICKET_K05}

Reasoning:"""

cot_response = chat(
    messages=[{'role': 'user', 'content': cot_prompt}],
    temperature=0.0,
)

print('\u2500\u2500 Stage 3: Chain-of-Thought \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Ticket [K05]: {TICKET_K05}')
print()
print('Prompt asks for step-by-step reasoning before the final answer.')
print()
print('LLM Reasoning Trace:')
print(cot_response)
print()
print('Improvement: We can now audit WHY the model chose this category.')
print('Useful for validating uncertain or ambiguous tickets.')

── Stage 3: Chain-of-Thought ──────────────────────────────────────────────
Ticket [K05]: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan

Prompt asks for step-by-step reasoning before the final answer.

LLM Reasoning Trace:
# Industrial Ticket Categorization

## Step-by-Step Analysis:

### 1. Main Subject of the Ticket
**Subject:** A procedural/compliance issue regarding worker safety protocols
- The ticket describes a situation where a technician entered a cleaning tank without proper safety procedures being followed

### 2. Keywords Indicating Category
- **"Prosedur LOTO"** (LOTO procedure) - Lockout/Tagout is a critical safety protocol
- **"belum dilakukan"** (not yet performed/not done)
- **"sebelum teknisi masuk"** (before technician enters)
- **"tangki pembersihan"** (cleaning tank) - confined space hazard

These keywords clearly point to a **safety compliance violation**.

### 3. Final Category

**SAFETY**

---

## Summary
This ticket reports a s

In [21]:
# CELL 3-D: System prompt with JSON output — production-ready format

TRIAGE_SYSTEM = """You are an industrial support triage assistant for a manufacturing plant.
Your job is to categorize maintenance and support tickets.

Categories:
- Mechanical: equipment failures, vibration, bearing, pump, motor, valve issues
- SAP/ERP: SAP transactions, purchase orders, goods receipt, user access, workflow
- Network/IT: server, network, connectivity, SCADA, printer, VPN, database issues
- Safety: near misses, APD violations, LOTO, permits, spills, fire alarms

Priority levels:
- Critical: immediate safety risk or production stoppage
- High: impacts production within 24 hours
- Medium: impacts operations but workaround exists
- Low: no immediate operational impact

Respond ONLY with valid JSON in this exact format:
{\"category\": \"...\", \"priority\": \"...\", \"reason\": \"one sentence explanation\"}"""

system_response = chat(
    messages=[{'role': 'user', 'content': f'Ticket: {TICKET_K05}'}],
    system=TRIAGE_SYSTEM,
    temperature=0.0,
)

print('\u2500\u2500 Stage 4: System Prompt + JSON Output \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Ticket [K05]: {TICKET_K05}')
print()
print('Raw LLM Response:')
print(system_response)
print()

# Parse JSON
try:
    clean = system_response.strip().strip('```json').strip('```').strip()
    result = json.loads(clean)
    print('Parsed Result:')
    print(f'  Category:  {result["category"]}')
    print(f'  Priority:  {result["priority"]}')
    print(f'  Reason:    {result["reason"]}')
    print()
    print('Improvement: Output is machine-parseable. Ready for database insertion.')
except json.JSONDecodeError as e:
    print(f'JSON parse error: {e}')
    print('Tip: Add more explicit format instructions or use function calling (Section 4).')

── Stage 4: System Prompt + JSON Output ─────────────────────────────────
Ticket [K05]: Prosedur LOTO belum dilakukan sebelum teknisi masuk ke dalam tangki pembersihan

Raw LLM Response:
```json
{"category": "Safety", "priority": "Critical", "reason": "Missing LOTO procedure before technician entry into cleaning tank poses immediate risk of serious injury or death from unexpected equipment startup or hazardous substance exposure."}
```

Parsed Result:
  Category:  Safety
  Priority:  Critical
  Reason:    Missing LOTO procedure before technician entry into cleaning tank poses immediate risk of serious injury or death from unexpected equipment startup or hazardous substance exposure.

Improvement: Output is machine-parseable. Ready for database insertion.


---
## Section 4 — Function Calling: Guaranteed Structured Output

Function calling (also called tool use) is a mechanism where the model is given a **schema** describing a function it can call, and it returns a structured object matching that schema instead of free text.

Think of it as a contract:
- **You define:** the function name, description, and parameter schema (JSON Schema format)
- **The model returns:** a `tool_use` block with parameter values guaranteed to match your schema
- **You execute:** whatever logic you want with those validated, typed parameters

### When to use function calling vs. JSON system prompt

| Approach | Pros | Cons |
|---|---|---|
| JSON system prompt | Works on all providers | Model may deviate; need robust parsing |
| Function calling | Schema-validated output | Requires API support (Anthropic, OpenAI) |

### Ollama fallback

Ollama models (Llama 3.2, Qwen2.5) do not support native tool use. When `PROVIDER == 'ollama'`, we fall back to the JSON system prompt approach from Section 3. The parsing logic is identical — participants see the same output structure regardless of provider.

### Design decision: enum constraints

The `category` and `priority` parameters use `enum` in the JSON Schema. This tells the model:
- It must return exactly one of the listed values
- No free-text improvisation is allowed
- The output can be used directly as a database enum value

In [22]:
# CELL 4-A: Function calling (Anthropic/OpenAI) or JSON fallback (Ollama/Mock)

TICKET_M01 = df[df['id'] == 'M01']['text'].values[0]

def categorize_with_tools(ticket_text):
    """Categorize a ticket using function calling or JSON fallback."""

    if PROVIDER == 'anthropic':
        tool_schema = {
            'name': 'categorize_ticket',
            'description': 'Categorize an industrial support ticket with priority and action required.',
            'input_schema': {
                'type': 'object',
                'properties': {
                    'category': {
                        'type': 'string',
                        'enum': ['Mechanical', 'SAP/ERP', 'Network/IT', 'Safety'],
                        'description': 'Ticket category'
                    },
                    'priority': {
                        'type': 'string',
                        'enum': ['Critical', 'High', 'Medium', 'Low'],
                        'description': 'Urgency level'
                    },
                    'equipment_id': {
                        'type': 'string',
                        'description': 'Equipment ID mentioned (e.g. P-101, GB-103) or null if none'
                    },
                    'action_required': {
                        'type': 'string',
                        'description': 'Concise action required to resolve the ticket'
                    }
                },
                'required': ['category', 'priority', 'action_required']
            }
        }

        response = client.messages.create(
            model=DEFAULT_MODEL,
            max_tokens=512,
            tools=[tool_schema],
            tool_choice={'type': 'auto'},
            messages=[{'role': 'user', 'content': f'Categorize this industrial ticket: {ticket_text}'}],
            system='You are an industrial support triage assistant.',
            temperature=0.0,
        )

        for block in response.content:
            if block.type == 'tool_use':
                return block.input
        return {}

    elif PROVIDER == 'openai':
        functions = [{
            'name': 'categorize_ticket',
            'description': 'Categorize an industrial support ticket.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'category': {'type': 'string', 'enum': ['Mechanical', 'SAP/ERP', 'Network/IT', 'Safety']},
                    'priority': {'type': 'string', 'enum': ['Critical', 'High', 'Medium', 'Low']},
                    'equipment_id': {'type': 'string'},
                    'action_required': {'type': 'string'},
                },
                'required': ['category', 'priority', 'action_required']
            }
        }]

        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=[
                {'role': 'system', 'content': 'You are an industrial support triage assistant.'},
                {'role': 'user', 'content': f'Categorize this industrial ticket: {ticket_text}'}
            ],
            functions=functions,
            function_call={'name': 'categorize_ticket'},
            temperature=0.0,
        )
        fn_args = response.choices[0].message.function_call.arguments
        return json.loads(fn_args)

    elif PROVIDER == 'mock':
        system = ('You are an industrial triage assistant. '
                  'Respond ONLY with valid JSON: '
                  '{"category": "Mechanical|SAP/ERP|Network/IT|Safety", '
                  '"priority": "Critical|High|Medium|Low", '
                  '"equipment_id": "...", "action_required": "..."}')
        result = _mock_llm(
            messages=[{'role': 'user', 'content': f'Categorize: {ticket_text}'}],
            system=system,
        )
        clean = result.strip().strip('```json').strip('```').strip()
        parsed = json.loads(clean)
        # Add a plausible action_required if missing
        if 'action_required' not in parsed:
            parsed['action_required'] = 'Investigate and resolve per standard procedure.'
        return parsed

    else:  # ollama fallback
        system = ('You are an industrial triage assistant. '
                  'Respond ONLY with valid JSON: '
                  '{"category": "Mechanical|SAP/ERP|Network/IT|Safety", '
                  '"priority": "Critical|High|Medium|Low", '
                  '"equipment_id": "...", "action_required": "..."}')
        result = chat(
            messages=[{'role': 'user', 'content': f'Categorize: {ticket_text}'}],
            system=system,
            temperature=0.0,
        )
        clean = result.strip().strip('```json').strip('```').strip()
        return json.loads(clean)


# ── Run on ticket M01 ───────────────────────────────────────────────────────────────────────
result = categorize_with_tools(TICKET_M01)

print('\u2500\u2500 Function Calling Result \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Ticket [M01]: {TICKET_M01}')
print()
print(f'Category:        {result.get("category")}')
print(f'Priority:        {result.get("priority")}')
print(f'Equipment ID:    {result.get("equipment_id", "N/A")}')
print(f'Action Required: {result.get("action_required")}')
print()
if PROVIDER in ['anthropic', 'openai']:
    mode_label = 'native tool use'
elif PROVIDER == 'mock':
    mode_label = 'mock keyword fallback'
else:
    mode_label = 'JSON fallback'
print(f'Provider used: {PROVIDER}  ({mode_label})')


── Function Calling Result ────────────────────────────────────────────────
Ticket [M01]: Pompa sentrifugal P-101 vibrasi berlebihan, perlu ganti bearing segera sebelum shutdown

Category:        Mechanical
Priority:        Critical
Equipment ID:    P-101
Action Required: Replace bearing on centrifugal pump P-101 immediately to prevent equipment shutdown

Provider used: anthropic  (native tool use)


---
## Section 5 — Model Comparison: The Accuracy/Latency/Cost Triangle

No model wins on all three dimensions. Choosing the right model is an engineering trade-off:

| Model | Strengths | Best for |
|---|---|---|
| **Claude Haiku 4.5** | Fastest, cheapest, multilingual | High-volume triage, dev/test |
| **Claude Sonnet 4.6** | Best reasoning, 200k context | Complex extraction, low-volume |
| **GPT-3.5 Turbo** | Low cost, fast | High-volume English tasks |
| **GPT-4o** | Multimodal, strong reasoning | Vision, complex docs |
| **Llama 3.2 (Ollama)** | Free, local, private | Air-gapped environments |
| **Qwen2.5 (Ollama)** | Best free multilingual | Indonesian-heavy workloads |

### The decision framework

1. **Is accuracy critical?** → Start with the best model, optimize down later
2. **Is volume high?** → Use the cheapest model that meets accuracy threshold
3. **Is data sensitive?** → Use local Ollama model (no data leaves your network)
4. **Is latency a constraint?** → Measure first; Haiku is typically <1s per call

### Design decision: time.time() for latency measurement

`time.time()` returns the Unix epoch as a float (seconds since 1970). Subtracting before/after an API call gives wall-clock latency, which includes network round-trip + model processing time. This is the most relevant metric for production systems.

In [23]:
# CELL 5-A: Benchmark 5 tickets — category, latency, token usage

BENCHMARK_IDS = ['M01', 'S01', 'N01', 'K05', 'M07']
benchmark_tickets = df[df['id'].isin(BENCHMARK_IDS)].reset_index(drop=True)

results = []

for _, row in benchmark_tickets.iterrows():
    t0 = time.time()

    response_text = chat(
        messages=[{'role': 'user', 'content': f'Ticket: {row["text"]}'}],
        system=TRIAGE_SYSTEM,
        temperature=0.0,
    )

    latency_ms = int((time.time() - t0) * 1000)

    try:
        clean = response_text.strip().strip('```json').strip('```').strip()
        parsed = json.loads(clean)
        category = parsed.get('category', 'Parse error')
        priority = parsed.get('priority', '?')
    except Exception:
        category = response_text[:40].strip()
        priority = '?'

    input_tok_est = len(enc.encode(TRIAGE_SYSTEM + row['text']))
    output_tok_est = len(enc.encode(response_text))

    results.append({
        'ticket_id': row['id'],
        'true_category': row['category'],
        'llm_category': category,
        'priority': priority,
        'latency_ms': latency_ms,
        'input_tokens': input_tok_est,
        'output_tokens': output_tok_est,
        'correct': category.strip() == row['category'].strip(),
    })

bench_df = pd.DataFrame(results)

print('\u2500\u2500 Benchmark Results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Provider: {PROVIDER}  |  Model: {DEFAULT_MODEL}')
print()
print(bench_df[['ticket_id', 'true_category', 'llm_category', 'priority', 'latency_ms', 'input_tokens', 'output_tokens']].to_string(index=False))
print()
print(f'Accuracy:       {bench_df["correct"].mean():.0%}  ({bench_df["correct"].sum()}/{len(bench_df)})')
print(f'Avg latency:    {bench_df["latency_ms"].mean():.0f} ms')
print(f'Avg in tokens:  {bench_df["input_tokens"].mean():.0f}')
print(f'Avg out tokens: {bench_df["output_tokens"].mean():.0f}')

── Benchmark Results ────────────────────────────────────────────────────
Provider: anthropic  |  Model: claude-haiku-4-5-20251001

ticket_id true_category llm_category priority  latency_ms  input_tokens  output_tokens
      M01    Mechanical   Mechanical Critical        1185           199             50
      M07    Mechanical   Mechanical     High        1001           195             46
      S01       SAP/ERP      SAP/ERP   Medium        2019           190             55
      N01    Network/IT   Network/IT Critical        1208           189             50
      K05        Safety       Safety Critical        1362           193             53

Accuracy:       100%  (5/5)
Avg latency:    1355 ms
Avg in tokens:  193
Avg out tokens: 51


---
## Section 6 — Full Triage Pipeline: All 40 Tickets

This is the payoff section. We run all 40 industrial tickets through our best prompt strategy (system prompt + JSON output) and compare the LLM's categories against:
1. The ground truth labels (in `df['category']`)
2. Day 1's K-Means clustering (silhouette score: 0.027)

### Why the LLM should dramatically outperform K-Means here

K-Means works on TF-IDF vectors — it can only find clusters based on **shared vocabulary**. When the same words ("perlu", "segera", "cek") appear across all categories, K-Means cannot distinguish them.

The LLM understands:
- **Semantics** — "vibrasi berlebihan" and "excessive vibration" mean the same thing
- **Indonesian domain knowledge** — "LOTO" = Lockout/Tagout = Safety
- **Context** — "server" in SCADA context = Network/IT; "server" in SAP context = SAP/ERP

### Design decision: rate limiting delay

`time.sleep(0.1)` between calls inserts a 100ms pause. This prevents burst-rate issues on restricted API keys and makes the progress printout readable.

### Design decision: list-of-dicts pattern

We collect results into a list of dicts and convert once at the end (`pd.DataFrame(results)`) rather than growing a DataFrame row-by-row. List append + single conversion is significantly faster for 40+ rows.

In [24]:
# CELL 6-A: Run all 40 tickets through the LLM triage pipeline

print('Running all 40 tickets through LLM triage pipeline...')
print(f'Provider: {PROVIDER}  |  Model: {DEFAULT_MODEL}')
print()

pipeline_results = []

for i, row in df.iterrows():
    t0 = time.time()

    response_text = chat(
        messages=[{'role': 'user', 'content': f'Ticket: {row["text"]}'}],
        system=TRIAGE_SYSTEM,
        temperature=0.0,
    )

    latency_ms = int((time.time() - t0) * 1000)

    try:
        clean = response_text.strip().strip('```json').strip('```').strip()
        parsed = json.loads(clean)
        llm_cat = parsed.get('category', 'Unknown')
        llm_pri = parsed.get('priority', 'Unknown')
    except Exception:
        llm_cat = 'ParseError'
        llm_pri = 'Unknown'

    pipeline_results.append({
        'id': row['id'],
        'true_category': row['category'],
        'llm_category': llm_cat,
        'llm_priority': llm_pri,
        'latency_ms': latency_ms,
        'correct': llm_cat.strip() == row['category'].strip(),
    })

    status = '\u2713' if llm_cat.strip() == row['category'].strip() else '\u2717'
    print(f'  [{row["id"]}] {status} LLM={llm_cat:<12} True={row["category"]:<12} ({latency_ms}ms)')

    time.sleep(0.1)

pipeline_df = pd.DataFrame(pipeline_results)

print()
print('\u2500\u2500 Pipeline Summary \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'Total tickets:     {len(pipeline_df)}')
print(f'Correct:           {pipeline_df["correct"].sum()} / {len(pipeline_df)}')
print(f'Accuracy:          {pipeline_df["correct"].mean():.1%}')
print(f'Avg latency:       {pipeline_df["latency_ms"].mean():.0f} ms')
print(f'Total elapsed:     ~{pipeline_df["latency_ms"].sum()/1000:.0f}s')

Running all 40 tickets through LLM triage pipeline...
Provider: anthropic  |  Model: claude-haiku-4-5-20251001

  [M01] ✓ LLM=Mechanical   True=Mechanical   (1015ms)
  [M02] ✓ LLM=Mechanical   True=Mechanical   (1220ms)
  [M03] ✓ LLM=Mechanical   True=Mechanical   (965ms)
  [M04] ✓ LLM=Mechanical   True=Mechanical   (1366ms)
  [M05] ✓ LLM=Mechanical   True=Mechanical   (1198ms)
  [M06] ✓ LLM=Mechanical   True=Mechanical   (1132ms)
  [M07] ✓ LLM=Mechanical   True=Mechanical   (982ms)
  [M08] ✓ LLM=Mechanical   True=Mechanical   (1608ms)
  [M09] ✓ LLM=Mechanical   True=Mechanical   (1401ms)
  [M10] ✓ LLM=Mechanical   True=Mechanical   (1159ms)
  [S01] ✓ LLM=SAP/ERP      True=SAP/ERP      (1129ms)
  [S02] ✓ LLM=SAP/ERP      True=SAP/ERP      (1164ms)
  [S03] ✓ LLM=SAP/ERP      True=SAP/ERP      (2561ms)
  [S04] ✓ LLM=SAP/ERP      True=SAP/ERP      (1047ms)
  [S05] ✓ LLM=SAP/ERP      True=SAP/ERP      (966ms)
  [S06] ✓ LLM=SAP/ERP      True=SAP/ERP      (1931ms)
  [S07] ✓ LLM=SAP/ERP      

In [25]:
# CELL 6-B: Crosstab LLM vs. ground truth + compare with Day 1 K-Means

print('\u2500\u2500 LLM Category vs. Ground Truth \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
crosstab = pd.crosstab(
    pipeline_df['true_category'],
    pipeline_df['llm_category'],
    rownames=['True Category'],
    colnames=['LLM Category']
)
print(crosstab)
print()
print('Diagonal = correct assignments. Off-diagonal = misclassifications.')

print()
print('\u2500\u2500 Per-Category Accuracy \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
for cat in ['Mechanical', 'SAP/ERP', 'Network/IT', 'Safety']:
    cat_df = pipeline_df[pipeline_df['true_category'] == cat]
    acc = cat_df['correct'].mean()
    print(f'  {cat:<15}: {acc:.0%}  ({cat_df["correct"].sum()}/{len(cat_df)})')

print()
print('\u2500\u2500 Comparison: LLM vs. Day 1 K-Means \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Day 1 K-Means silhouette score: 0.027  (low \u2014 overlapping vocabulary)')
print(f'  Day 1 K-Means visual accuracy:  ~30\u201340% (many misassignments across categories)')
print(f'  Day 2 LLM accuracy:             {pipeline_df["correct"].mean():.1%}  (understands semantics + Indonesian)')
print()
print('Why the LLM wins:')
print('  \u2713 Understands Bahasa Indonesia natively')
print('  \u2713 Knows domain context: LOTO = Safety, ME21N = SAP/ERP, SCADA = Network/IT')
print('  \u2713 No training required \u2014 zero-shot semantic understanding')
print()
print('Where K-Means is still useful:')
print('  \u2713 No API cost \u2014 runs offline, free')
print('  \u2713 Finds unexpected clusters in unlabelled data')
print('  \u2713 Does not require internet or external services')

── LLM Category vs. Ground Truth ────────────────────────────────────────────
LLM Category   Mechanical  Network/IT  SAP/ERP  Safety
True Category                                         
Mechanical             10           0        0       0
Network/IT              0          10        0       0
SAP/ERP                 0           1        9       0
Safety                  0           0        0      10

Diagonal = correct assignments. Off-diagonal = misclassifications.

── Per-Category Accuracy ───────────────────────────────────────────────────
  Mechanical     : 100%  (10/10)
  SAP/ERP        : 90%  (9/10)
  Network/IT     : 100%  (10/10)
  Safety         : 100%  (10/10)

── Comparison: LLM vs. Day 1 K-Means ──────────────────────────────────────────
  Day 1 K-Means silhouette score: 0.027  (low — overlapping vocabulary)
  Day 1 K-Means visual accuracy:  ~30–40% (many misassignments across categories)
  Day 2 LLM accuracy:             97.5%  (understands semantics + Indonesian)

Wh

---
## Section 7 — Next Steps & Reflection

### What we built tonight

Starting from a multi-provider client, we built a complete LLM triage pipeline:

1. **Detected** the LLM provider automatically (Anthropic → OpenAI → Ollama)
2. **Measured** token counts and API costs across 4 providers
3. **Made** our first LLM API call with system/user message roles
4. **Applied** 4 prompt engineering strategies: zero-shot, few-shot, CoT, system prompt
5. **Used** function calling to get schema-validated structured output
6. **Benchmarked** accuracy, latency, and token usage across 5 tickets
7. **Ran** all 40 Day 1 tickets through the LLM — comparing against K-Means

**Generated outputs:**
- `pipeline_df` — full results DataFrame (40 tickets × LLM labels)
- Crosstab: LLM category vs. ground truth

---

### How this connects to Days 3–5

| Day | Building on tonight | New capability |
|---|---|---|
| **Day 3** | System prompts + JSON output | Extract entities (equipment IDs, dates, failure modes) with NER prompts |
| **Day 4** | Semantic understanding | Add vector DB → semantic search over ticket history (RAG) |
| **Day 5** | Full LLM pipeline | Wrap in a conversational Q&A chatbot with memory |

---

### Try-it-yourself (homework)

1. **Switch models** — Change `DEFAULT_MODEL` to `claude-sonnet-4-6` and re-run Section 6.  
   Does accuracy improve? By how much? What is the cost difference?

2. **Improve the system prompt** — Add specific examples to `TRIAGE_SYSTEM` for ambiguous tickets.  
   Which tickets does the model misclassify? Can a better prompt fix them?

3. **Add a 5th category** — Extend the schema to include `"Environmental"` (spill cleanup, emissions).  
   Write 3 sample tickets. Does the model categorize them correctly zero-shot?

4. **Measure cost exactly** — For Anthropic users: modify the pipeline to collect `response.usage.input_tokens`  
   and `response.usage.output_tokens` per call. What is the actual cost vs. the estimate in Section 1?

5. **Build a simple ticket router** — Using the LLM pipeline output, write a function that:  
   - Takes a raw ticket text as input  
   - Returns: `{category, priority, assigned_team_email}`  
   - Maps each category to a fictional team email address  
   This is the foundation of an automated support routing system.

---

*Day 2 — LLMs Deep Dive | Industrial AI & LLM Training Program*